# Chapter 4.8: Multi-Objective Ranking

## Learning Objectives

By the end of this notebook, you will be able to:

1. Formulate **multi-objective ranking** problems with CTR, CVR, engagement, and satisfaction objectives
2. Implement **scalarization** methods: weighted sum, constraint-based, and Chebyshev
3. Explore the **Pareto front** for multi-objective ranking
4. Build **dynamic weight adjustment** systems with adaptive scalarization
5. Analyze **engagement vs satisfaction** trade-offs in practice
6. Implement multi-task architectures for multi-objective optimization
7. Design Pareto-optimal multi-objective ranking systems

## Prerequisites

- Understanding of multi-task learning
- Familiarity with CTR/CVR prediction
- Basic knowledge of optimization theory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part4/chapter_4.8_multi_objective.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part4/chapter_4.8_multi_objective.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple
import copy

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cpu')

## 1. The Multi-Objective Ranking Problem

Modern recommendation systems must optimize for multiple objectives simultaneously:

| Objective | Description | Business Goal |
|-----------|-------------|---------------|
| **CTR** | Click-through rate | User engagement |
| **CVR** | Conversion rate | Revenue |
| **Dwell time** | Time spent on content | User satisfaction |
| **Like rate** | Explicit positive feedback | Content quality |
| **Diversity** | Variety of recommendations | Long-term retention |

### The Challenge

These objectives often **conflict**: optimizing for clicks may hurt satisfaction (clickbait), while optimizing for conversions may reduce engagement (too many ads).

### Pareto Optimality

A solution is **Pareto optimal** if no objective can be improved without worsening another:

$$\nexists \theta' : f_i(\theta') \geq f_i(\theta) \ \forall i, \text{ and } \exists j : f_j(\theta') > f_j(\theta)$$

> **💡 Concept:** The Pareto front represents the set of all Pareto-optimal solutions. The business must choose a point on this front that best satisfies their priorities.

In [ ]:
def generate_multi_objective_data(num_samples=10000, num_features=8,
                                   vocab_sizes=None, seed=42):
    """Generate synthetic data with multiple objectives.
    
    Objectives: CTR, CVR, dwell_time, like_rate
    These objectives have controlled correlations and conflicts.
    """
    np.random.seed(seed)
    if vocab_sizes is None:
        vocab_sizes = [100, 200, 50, 300, 80, 150, 60, 120]
    
    sparse = np.column_stack([
        np.random.randint(0, vs, size=num_samples) for vs in vocab_sizes
    ])
    dense = np.random.randn(num_samples, 5).astype(np.float32)
    
    # Latent user/item quality factors
    attractiveness = 0.5 * dense[:, 0] + 0.3 * (sparse[:, 0] % 10) / 10
    quality = 0.4 * dense[:, 1] + 0.2 * dense[:, 2]
    commercial = 0.3 * dense[:, 3] + 0.2 * (sparse[:, 1] % 10) / 10
    
    # CTR: driven by attractiveness (can be clickbait)
    ctr_logit = attractiveness + 0.2 * quality + np.random.randn(num_samples) * 0.3
    ctr = (np.random.rand(num_samples) < 1 / (1 + np.exp(-ctr_logit))).astype(np.float32)
    
    # CVR (given click): driven by commercial intent and quality
    cvr_logit = commercial + 0.3 * quality - 0.5 + np.random.randn(num_samples) * 0.3
    cvr = (np.random.rand(num_samples) < 1 / (1 + np.exp(-cvr_logit))).astype(np.float32)
    cvr = cvr * ctr  # CVR only matters if clicked
    
    # Dwell time (satisfaction proxy): driven by quality, hurt by clickbait
    dwell_logit = quality - 0.3 * (attractiveness - quality).clip(0) + np.random.randn(num_samples) * 0.3
    dwell_time = np.clip(np.exp(dwell_logit), 0, 10).astype(np.float32)
    dwell_time = dwell_time * ctr  # Only if clicked
    
    # Like rate: driven by quality
    like_logit = 0.8 * quality + 0.1 * attractiveness - 0.5 + np.random.randn(num_samples) * 0.3
    like = (np.random.rand(num_samples) < 1 / (1 + np.exp(-like_logit))).astype(np.float32)
    like = like * ctr  # Only if clicked
    
    return {
        'sparse': torch.LongTensor(sparse),
        'dense': torch.FloatTensor(dense),
        'ctr': torch.FloatTensor(ctr),
        'cvr': torch.FloatTensor(cvr),
        'dwell': torch.FloatTensor(dwell_time),
        'like': torch.FloatTensor(like),
    }, vocab_sizes


data, VOCAB_SIZES = generate_multi_objective_data()
EMBED_DIM = 16

print("Objective statistics:")
for obj in ['ctr', 'cvr', 'like']:
    print(f"  {obj}: mean={data[obj].mean():.3f}")
print(f"  dwell: mean={data['dwell'].mean():.3f}, max={data['dwell'].max():.3f}")

# Show correlations between objectives
print("\nObjective correlations:")
objectives = np.column_stack([data[k].numpy() for k in ['ctr', 'cvr', 'dwell', 'like']])
corr = np.corrcoef(objectives.T)
obj_names = ['CTR', 'CVR', 'Dwell', 'Like']
print(f"{'':>8}", ''.join(f'{n:>8}' for n in obj_names))
for i, name in enumerate(obj_names):
    print(f"{name:>8}", ''.join(f'{corr[i,j]:>8.3f}' for j in range(4)))

## 2. Multi-Task Model Architecture

We use a shared-bottom architecture with task-specific towers, following the MMoE pattern.

In [ ]:
class MultiObjectiveModel(nn.Module):
    """Multi-objective ranking model with shared bottom and task-specific towers."""
    def __init__(self, vocab_sizes, embed_dim, num_dense=5,
                 num_objectives=4, hidden_dim=128, num_experts=4):
        super().__init__()
        self.num_objectives = num_objectives
        self.embeddings = nn.ModuleList([nn.Embedding(vs, embed_dim) for vs in vocab_sizes])
        
        input_dim = embed_dim * len(vocab_sizes) + num_dense
        
        # Shared experts (MMoE style)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            ) for _ in range(num_experts)
        ])
        
        # Task-specific gates
        self.gates = nn.ModuleList([
            nn.Linear(input_dim, num_experts) for _ in range(num_objectives)
        ])
        
        # Task-specific towers
        self.towers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 1)
            ) for _ in range(num_objectives)
        ])
    
    def forward(self, sparse, dense):
        embeds = [self.embeddings[i](sparse[:, i]) for i in range(sparse.size(1))]
        x = torch.cat(embeds + [dense], dim=-1)
        
        # Expert outputs
        expert_outputs = torch.stack([e(x) for e in self.experts], dim=1)  # (batch, experts, hidden)
        
        # Task-specific outputs
        outputs = []
        for t in range(self.num_objectives):
            gate_weights = F.softmax(self.gates[t](x), dim=-1)  # (batch, experts)
            task_input = torch.einsum('be,bed->bd', gate_weights, expert_outputs)
            outputs.append(self.towers[t](task_input).squeeze(-1))
        
        return outputs  # List of (batch,) tensors


model = MultiObjectiveModel(VOCAB_SIZES, EMBED_DIM)
outputs = model(data['sparse'][:32], data['dense'][:32])
print(f"Number of objectives: {len(outputs)}")
print(f"Each output shape: {outputs[0].shape}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Scalarization Methods

Scalarization converts a multi-objective problem into a single-objective one.

### 3.1 Weighted Sum
$$\mathcal{L} = \sum_i w_i \mathcal{L}_i$$

### 3.2 Chebyshev (Min-Max)
$$\mathcal{L} = \max_i w_i (\mathcal{L}_i - z_i^*)$$

where $z_i^*$ is the ideal point (best achievable for objective $i$).

### 3.3 Constraint-based
$$\min \mathcal{L}_1 \quad \text{s.t.} \quad \mathcal{L}_i \leq c_i, \, i > 1$$

> **⚠️ Common Pitfall:** Weighted sum scalarization can only find solutions on the convex part of the Pareto front. For non-convex Pareto fronts, Chebyshev scalarization is preferred.

In [ ]:
class WeightedSumScalarizer:
    """Weighted sum of task losses."""
    def __init__(self, weights: List[float]):
        self.weights = weights
    
    def __call__(self, losses: List[torch.Tensor]) -> torch.Tensor:
        return sum(w * l for w, l in zip(self.weights, losses))


class ChebyshevScalarizer:
    """Chebyshev (min-max) scalarization."""
    def __init__(self, weights: List[float], ideal_point: List[float] = None):
        self.weights = weights
        self.ideal_point = ideal_point or [0.0] * len(weights)
    
    def __call__(self, losses: List[torch.Tensor]) -> torch.Tensor:
        weighted_diffs = [
            w * (l - z) for w, l, z in zip(self.weights, losses, self.ideal_point)
        ]
        return torch.stack(weighted_diffs).max()


class ConstraintScalarizer:
    """Constraint-based scalarization.
    
    Optimize primary objective with soft constraints on others.
    """
    def __init__(self, primary_idx: int = 0, constraints: Dict[int, float] = None,
                 penalty_weight: float = 10.0):
        self.primary_idx = primary_idx
        self.constraints = constraints or {}
        self.penalty_weight = penalty_weight
    
    def __call__(self, losses: List[torch.Tensor]) -> torch.Tensor:
        total = losses[self.primary_idx]
        for idx, threshold in self.constraints.items():
            violation = F.relu(losses[idx] - threshold)
            total = total + self.penalty_weight * violation
        return total

In [ ]:
def compute_task_losses(model_outputs, labels_dict):
    """Compute individual task losses."""
    obj_names = ['ctr', 'cvr', 'dwell', 'like']
    losses = []
    for i, name in enumerate(obj_names):
        if name == 'dwell':
            loss = F.mse_loss(torch.sigmoid(model_outputs[i]) * 10, labels_dict[name])
        else:
            loss = F.binary_cross_entropy_with_logits(model_outputs[i], labels_dict[name])
        losses.append(loss)
    return losses


def train_multi_objective(model, data, scalarizer, epochs=15, batch_size=256, lr=1e-3):
    """Train multi-objective model with a given scalarizer."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(data['ctr'])
    obj_names = ['ctr', 'cvr', 'dwell', 'like']
    
    history = {name: [] for name in obj_names}
    history['total'] = []
    
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        epoch_losses = {name: 0.0 for name in obj_names}
        epoch_total = 0.0
        count = 0
        
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            outputs = model(data['sparse'][idx], data['dense'][idx])
            labels = {k: data[k][idx] for k in obj_names}
            
            task_losses = compute_task_losses(outputs, labels)
            total_loss = scalarizer(task_losses)
            
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            for j, name in enumerate(obj_names):
                epoch_losses[name] += task_losses[j].item()
            epoch_total += total_loss.item()
            count += 1
        
        for name in obj_names:
            history[name].append(epoch_losses[name] / count)
        history['total'].append(epoch_total / count)
    
    return history


# Compare scalarization strategies
scalarizers = {
    'Equal Weights': WeightedSumScalarizer([0.25, 0.25, 0.25, 0.25]),
    'CTR-focused': WeightedSumScalarizer([0.6, 0.2, 0.1, 0.1]),
    'Quality-focused': WeightedSumScalarizer([0.1, 0.1, 0.4, 0.4]),
    'Chebyshev': ChebyshevScalarizer([0.25, 0.25, 0.25, 0.25]),
}

all_histories = {}
for name, scalarizer in scalarizers.items():
    torch.manual_seed(42)
    model = MultiObjectiveModel(VOCAB_SIZES, EMBED_DIM)
    history = train_multi_objective(model, data, scalarizer)
    all_histories[name] = history
    print(f"{name:20s} | CTR: {history['ctr'][-1]:.4f} | CVR: {history['cvr'][-1]:.4f} | "
          f"Dwell: {history['dwell'][-1]:.4f} | Like: {history['like'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
obj_names = ['ctr', 'cvr', 'dwell', 'like']
titles = ['CTR Loss', 'CVR Loss', 'Dwell Time Loss', 'Like Rate Loss']

for ax_idx, (ax, obj, title) in enumerate(zip(axes.flat, obj_names, titles)):
    for strategy, history in all_histories.items():
        ax.plot(history[obj], label=strategy, linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Multi-Objective Training: Loss per Objective', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Pareto Front Exploration

We can explore the Pareto front by training models with different weight combinations and identifying non-dominated solutions.

In [ ]:
def explore_pareto_front(data, num_points=20, epochs=10):
    """Explore Pareto front by sampling weight combinations."""
    results = []
    
    for trial in range(num_points):
        # Sample random weights from Dirichlet distribution
        np.random.seed(trial * 7 + 1)
        weights = np.random.dirichlet(np.ones(4))
        
        torch.manual_seed(42)
        model = MultiObjectiveModel(VOCAB_SIZES, EMBED_DIM)
        scalarizer = WeightedSumScalarizer(weights.tolist())
        history = train_multi_objective(model, data, scalarizer, epochs=epochs)
        
        final_losses = {
            'ctr': history['ctr'][-1],
            'cvr': history['cvr'][-1],
            'dwell': history['dwell'][-1],
            'like': history['like'][-1],
            'weights': weights.tolist()
        }
        results.append(final_losses)
    
    return results


def find_pareto_front(results, obj1='ctr', obj2='like'):
    """Find Pareto-optimal solutions for two objectives."""
    points = np.array([(r[obj1], r[obj2]) for r in results])
    n = len(points)
    is_pareto = np.ones(n, dtype=bool)
    
    for i in range(n):
        for j in range(n):
            if i != j:
                if points[j, 0] <= points[i, 0] and points[j, 1] <= points[i, 1]:
                    if points[j, 0] < points[i, 0] or points[j, 1] < points[i, 1]:
                        is_pareto[i] = False
                        break
    
    return is_pareto


pareto_results = explore_pareto_front(data, num_points=25, epochs=10)

# Plot Pareto front for CTR vs Like
is_pareto = find_pareto_front(pareto_results, 'ctr', 'like')
ctr_losses = [r['ctr'] for r in pareto_results]
like_losses = [r['like'] for r in pareto_results]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    [ctr_losses[i] for i in range(len(ctr_losses)) if not is_pareto[i]],
    [like_losses[i] for i in range(len(like_losses)) if not is_pareto[i]],
    c='gray', alpha=0.5, s=80, label='Dominated'
)
pareto_ctr = [ctr_losses[i] for i in range(len(ctr_losses)) if is_pareto[i]]
pareto_like = [like_losses[i] for i in range(len(like_losses)) if is_pareto[i]]
ax.scatter(pareto_ctr, pareto_like, c='red', s=120, label='Pareto Front', zorder=5)

# Connect Pareto front points
sorted_idx = np.argsort(pareto_ctr)
ax.plot([pareto_ctr[i] for i in sorted_idx], [pareto_like[i] for i in sorted_idx],
        'r--', alpha=0.5)

ax.set_xlabel('CTR Loss (lower = better CTR)')
ax.set_ylabel('Like Rate Loss (lower = better likes)')
ax.set_title('Pareto Front: CTR vs Like Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Dynamic Weight Adjustment

Instead of fixed weights, adaptive methods adjust weights during training based on task performance.

### 5.1 Uncertainty Weighting (Kendall et al., 2018)

Learn task weights based on homoscedastic uncertainty:

$$\mathcal{L} = \sum_i \frac{1}{2\sigma_i^2} \mathcal{L}_i + \log \sigma_i$$

### 5.2 GradNorm (Chen et al., 2018)

Normalize gradient magnitudes across tasks to balance training speed.

> **🔑 Pro Tip:** Uncertainty weighting works well when tasks have different scales (e.g., binary classification vs regression). GradNorm is better when tasks train at very different speeds.

In [ ]:
class UncertaintyWeighting(nn.Module):
    """Learnable task weights based on uncertainty."""
    def __init__(self, num_tasks: int):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(num_tasks))
    
    def forward(self, losses: List[torch.Tensor]) -> torch.Tensor:
        total = 0
        for i, loss in enumerate(losses):
            precision = torch.exp(-2 * self.log_sigma[i])
            total = total + precision * loss + self.log_sigma[i]
        return total
    
    def get_weights(self):
        """Get effective task weights."""
        return torch.exp(-2 * self.log_sigma).detach().numpy()


class DynamicWeightAveraging:
    """Dynamic Weight Averaging (DWA): adjust weights based on loss ratios."""
    def __init__(self, num_tasks: int, temperature: float = 2.0):
        self.num_tasks = num_tasks
        self.temperature = temperature
        self.prev_losses = None
    
    def __call__(self, losses: List[torch.Tensor],
                 current_losses: List[float] = None) -> torch.Tensor:
        if self.prev_losses is None or current_losses is None:
            weights = [1.0 / self.num_tasks] * self.num_tasks
        else:
            # Weight based on loss ratio (higher ratio = more weight)
            ratios = [c / max(p, 1e-8) for c, p in zip(current_losses, self.prev_losses)]
            exp_ratios = [np.exp(r / self.temperature) for r in ratios]
            total = sum(exp_ratios)
            weights = [self.num_tasks * r / total for r in exp_ratios]
        
        if current_losses is not None:
            self.prev_losses = current_losses
        
        return sum(w * l for w, l in zip(weights, losses))


# Train with uncertainty weighting
torch.manual_seed(42)
uw_model = MultiObjectiveModel(VOCAB_SIZES, EMBED_DIM)
uw = UncertaintyWeighting(4)

optimizer = torch.optim.Adam(
    list(uw_model.parameters()) + list(uw.parameters()), lr=1e-3
)

uw_history = {'ctr': [], 'cvr': [], 'dwell': [], 'like': [], 'weights': []}
obj_names = ['ctr', 'cvr', 'dwell', 'like']
n = len(data['ctr'])

for epoch in range(15):
    uw_model.train()
    perm = torch.randperm(n)
    epoch_losses = [0.0] * 4
    count = 0
    
    for i in range(0, n, 256):
        idx = perm[i:i+256]
        outputs = uw_model(data['sparse'][idx], data['dense'][idx])
        labels = {k: data[k][idx] for k in obj_names}
        task_losses = compute_task_losses(outputs, labels)
        
        total_loss = uw(task_losses)
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        for j in range(4):
            epoch_losses[j] += task_losses[j].item()
        count += 1
    
    for j, name in enumerate(obj_names):
        uw_history[name].append(epoch_losses[j] / count)
    uw_history['weights'].append(uw.get_weights().tolist())

print("Uncertainty Weighting - Learned task weights:")
final_weights = uw_history['weights'][-1]
for name, w in zip(obj_names, final_weights):
    print(f"  {name}: {w:.4f}")

In [ ]:
# Visualize weight evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

weights_over_time = np.array(uw_history['weights'])
for i, name in enumerate(obj_names):
    axes[0].plot(weights_over_time[:, i], label=name, linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Task Weight')
axes[0].set_title('Uncertainty Weighting: Weight Evolution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compare final losses
methods = ['Equal', 'CTR-focused', 'Quality-focused', 'Uncertainty']
final_data = {
    'Equal': [all_histories['Equal Weights'][k][-1] for k in obj_names],
    'CTR-focused': [all_histories['CTR-focused'][k][-1] for k in obj_names],
    'Quality-focused': [all_histories['Quality-focused'][k][-1] for k in obj_names],
    'Uncertainty': [uw_history[k][-1] for k in obj_names],
}

x = np.arange(len(obj_names))
width = 0.2
for i, method in enumerate(methods):
    axes[1].bar(x + i * width, final_data[method], width, label=method)

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels([n.upper() for n in obj_names])
axes[1].set_ylabel('Final Loss')
axes[1].set_title('Final Losses by Objective and Strategy')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Engagement vs Satisfaction Trade-offs

A critical trade-off in modern recommender systems: **short-term engagement** (clicks, views) often conflicts with **long-term satisfaction** (dwell time, likes, returns).

### The Clickbait Problem

Optimizing purely for CTR can lead to:
- Sensationalized content that gets clicks but disappoints users
- Filter bubbles that maximize short-term engagement
- User churn due to declining satisfaction

### Practical Solutions

1. **Quality gates**: Set minimum quality thresholds before optimizing CTR
2. **Satisfaction-weighted CTR**: Weight CTR by dwell time or satisfaction
3. **Multi-phase training**: First optimize quality, then tune for engagement

> **💡 Concept:** The optimal ranking function is often: $\text{score} = \text{pCTR}^{\alpha} \cdot \text{pDwell}^{\beta} \cdot \text{pLike}^{\gamma}$ where the exponents are tuned based on business priorities.

In [ ]:
def compute_ranking_score(model, data, alpha=1.0, beta=0.0, gamma=0.0):
    """Compute ranking score as weighted product of objective predictions."""
    model.eval()
    with torch.no_grad():
        outputs = model(data['sparse'], data['dense'])
        p_ctr = torch.sigmoid(outputs[0])
        p_dwell = torch.sigmoid(outputs[2])
        p_like = torch.sigmoid(outputs[3])
        
        score = p_ctr ** alpha * p_dwell ** beta * p_like ** gamma
    return score.numpy()


# Train a multi-objective model
torch.manual_seed(42)
combo_model = MultiObjectiveModel(VOCAB_SIZES, EMBED_DIM)
combo_scalarizer = WeightedSumScalarizer([0.3, 0.2, 0.3, 0.2])
train_multi_objective(combo_model, data, combo_scalarizer, epochs=15)

# Compare different scoring formulas
scoring_configs = [
    {'name': 'CTR only', 'alpha': 1.0, 'beta': 0.0, 'gamma': 0.0},
    {'name': 'CTR + Dwell', 'alpha': 1.0, 'beta': 0.5, 'gamma': 0.0},
    {'name': 'CTR + Like', 'alpha': 1.0, 'beta': 0.0, 'gamma': 0.5},
    {'name': 'Balanced', 'alpha': 1.0, 'beta': 0.3, 'gamma': 0.3},
    {'name': 'Quality-first', 'alpha': 0.5, 'beta': 0.5, 'gamma': 0.5},
]

print(f"{'Scoring Formula':<20} {'Avg CTR':>10} {'Avg Dwell':>10} {'Avg Like':>10}")
print("-" * 55)

for config in scoring_configs:
    scores = compute_ranking_score(
        combo_model, data, config['alpha'], config['beta'], config['gamma']
    )
    # Simulate: select top-20% by score
    threshold = np.percentile(scores, 80)
    selected = scores >= threshold
    
    avg_ctr = data['ctr'].numpy()[selected].mean()
    avg_dwell = data['dwell'].numpy()[selected].mean()
    avg_like = data['like'].numpy()[selected].mean()
    
    print(f"{config['name']:<20} {avg_ctr:>10.4f} {avg_dwell:>10.4f} {avg_like:>10.4f}")

## Exercises

### 🏋️ Exercise 1: Implement Pareto-Optimal Multi-Objective Ranking

Implement a training procedure that discovers diverse points on the Pareto front using different preference vectors.

In [ ]:
class ParetoMTL:
    """Pareto Multi-Task Learning (Lin et al., 2019).
    
    Finds a gradient direction that improves all tasks (Pareto improvement)
    or finds the closest direction to a given preference vector.
    
    Algorithm:
    1. Compute per-task gradients
    2. Find the minimum-norm point in the convex hull of gradients
    3. If the point has positive norm -> Pareto improvement exists
    4. Otherwise -> we are at the Pareto front, use preference vector
    
    TODO:
    1. Implement gradient computation per task
    2. Implement min-norm solver (Frank-Wolfe or QP)
    3. Apply the Pareto-optimal gradient update
    4. Train and collect Pareto front solutions
    """
    def __init__(self, num_tasks, preference=None):
        self.num_tasks = num_tasks
        self.preference = preference or [1.0/num_tasks] * num_tasks
    
    def find_pareto_gradient(self, task_gradients):
        # TODO: Find the min-norm point in convex hull of task gradients
        # This determines the Pareto-optimal update direction
        pass
    
    def step(self, model, task_losses):
        # TODO: Compute Pareto-optimal gradient and update model
        pass


# TODO: Train with ParetoMTL and visualize the discovered Pareto front

### 🏋️ Exercise 2: Implement GradNorm

Implement GradNorm for balancing multi-task gradient magnitudes.

In [ ]:
class GradNorm:
    """GradNorm: Gradient Normalization for multi-task learning.
    
    Dynamically adjusts task weights to equalize the gradient norms
    at a shared layer, accounting for different task learning speeds.
    
    Args:
        num_tasks: number of tasks
        alpha: asymmetry parameter (higher = more equalization)
    
    TODO:
    1. Track initial task losses for computing relative loss ratios
    2. Compute gradient norms at the shared layer per task
    3. Compute target gradient norms based on loss ratios
    4. Update task weights to match target norms
    5. Renormalize weights to sum to num_tasks
    """
    def __init__(self, num_tasks, alpha=1.5):
        self.num_tasks = num_tasks
        self.alpha = alpha
        self.weights = nn.Parameter(torch.ones(num_tasks))
        self.initial_losses = None
    
    def step(self, task_losses, shared_layer):
        # TODO: Implement GradNorm weight update
        pass


# TODO: Train with GradNorm and compare weight evolution against uncertainty weighting

### 🏋️ Exercise 3: Satisfaction-Constrained CTR Optimization

Implement a system that maximizes CTR subject to minimum dwell time and like rate constraints.

In [ ]:
def satisfaction_constrained_training(model, data, min_dwell=None, min_like=None,
                                       epochs=15, lr=1e-3):
    """Train to maximize CTR while maintaining satisfaction constraints.
    
    Uses Lagrangian relaxation:
    L = L_ctr + lambda_dwell * max(0, min_dwell - dwell_metric)
                + lambda_like * max(0, min_like - like_metric)
    
    Dual variables (lambdas) are updated via gradient ascent.
    
    TODO:
    1. Initialize Lagrange multipliers
    2. For each epoch:
       a. Train model with Lagrangian objective
       b. Update Lagrange multipliers based on constraint violations
    3. Track constraint satisfaction over training
    4. Plot: CTR vs dwell time vs like rate over training
    """
    pass


# TODO: Run constrained optimization and visualize results

### 🏋️ Exercise 4: A/B Test Simulation

Simulate an A/B test comparing different multi-objective strategies.

In [ ]:
def simulate_ab_test(models_dict, data, num_users=1000, items_per_user=10):
    """Simulate an A/B test for multi-objective ranking strategies.
    
    For each user:
    1. Score all items with each model
    2. Select top-k items per model
    3. Simulate user interactions based on true labels
    4. Compute business metrics: total clicks, conversions, satisfaction
    
    TODO:
    1. Implement user simulation
    2. Compute per-model metrics
    3. Compute statistical significance
    4. Visualize as a dashboard-style report
    """
    pass


# TODO: Run A/B test simulation with different strategies

## Summary

In this notebook, we explored multi-objective ranking optimization:

| Method | Key Idea | Advantages | Limitations |
|--------|----------|------------|-------------|
| **Weighted Sum** | Fixed linear combination | Simple, fast | Only convex front |
| **Chebyshev** | Min-max fairness | Full Pareto front | Sensitive to ideal point |
| **Constraint-based** | Lagrangian relaxation | Business-friendly | Need good thresholds |
| **Uncertainty weighting** | Learned uncertainty | Automatic balancing | May not respect priorities |
| **DWA** | Loss ratio-based | Adapts to training dynamics | Hyperparameter sensitive |
| **GradNorm** | Gradient magnitude equalization | Balances training speed | Requires shared layer |

**Key takeaways:**
- Multi-objective ranking requires explicit trade-off management
- No single scalarization method dominates; choose based on business needs
- Dynamic weight adjustment adapts to changing training dynamics
- The scoring formula $\text{pCTR}^\alpha \cdot \text{pQuality}^\beta$ provides flexible control
- Always measure engagement AND satisfaction metrics in production